In [6]:
import os
print(os.getcwd())

c:\Users\manue\Downloads\Proyectos\breast-cancer-wisconsin\notebooks


In [7]:
from sklearn.datasets import load_breast_cancer
import pandas as pd
from sklearn.model_selection import train_test_split

# Cargar el dataset de Wisconsin (breast cancer): 0 = maligno, 1 = benigno
data = load_breast_cancer()
df_data = pd.DataFrame(data.data, columns=data.feature_names)
df_target = pd.DataFrame(data.target, columns=['target'])

# 5% se reserva para "predecir" luego desde la app (sin la respuesta)
X_train, X_pred, y_train, y_pred = train_test_split(
    df_data, df_target, test_size=0.05, random_state=42
)

df_train = pd.concat([X_train, y_train], axis=1)
df_train.to_csv("../datasets/tabular/breast_cancer_train.csv", index=False)

df_pred = pd.DataFrame(X_pred, columns=df_data.columns)
df_pred.to_csv("../datasets/tabular/breast_cancer_pred.csv", index=False)

print(df_train.shape, df_pred.shape)

(540, 31) (29, 30)


In [8]:
import tensorflow as tf
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import numpy as np
import joblib

tf.keras.utils.set_random_seed(42)
tf.config.experimental.enable_op_determinism()

# Leer el CSV que acabas de crear (ya no usamos load_breast_cancer aquí)
df = pd.read_csv("../datasets/tabular/breast_cancer_train.csv")
X = df.drop(columns=["target"])
y = df["target"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Escalar: MUY importante en este dataset porque las columnas tienen escalas muy distintas
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Guardamos el scaler: el backend lo va a necesitar para escalar datos nuevos igual que aquí
joblib.dump(scaler, "../models/scaler.joblib")

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)

history = model.fit(X_train, y_train, epochs=50, batch_size=32, verbose=2)

test_loss, test_acc = model.evaluate(X_test, y_test)
print(f'\nTest Accuracy: {test_acc:.4f}')

probs = model.predict(X_test)
predicted = (probs > 0.5).astype(int).flatten()
print(classification_report(y_test, predicted, target_names=["maligno", "benigno"]))

Epoch 1/50
14/14 - 3s - 206ms/step - accuracy: 0.8750 - loss: 0.5401
Epoch 2/50
14/14 - 0s - 14ms/step - accuracy: 0.9421 - loss: 0.3496
Epoch 3/50
14/14 - 0s - 13ms/step - accuracy: 0.9468 - loss: 0.2173
Epoch 4/50
14/14 - 0s - 8ms/step - accuracy: 0.9630 - loss: 0.1405
Epoch 5/50
14/14 - 0s - 7ms/step - accuracy: 0.9769 - loss: 0.0995
Epoch 6/50
14/14 - 0s - 7ms/step - accuracy: 0.9861 - loss: 0.0776
Epoch 7/50
14/14 - 0s - 8ms/step - accuracy: 0.9838 - loss: 0.0651
Epoch 8/50
14/14 - 0s - 7ms/step - accuracy: 0.9861 - loss: 0.0572
Epoch 9/50
14/14 - 0s - 6ms/step - accuracy: 0.9861 - loss: 0.0518
Epoch 10/50
14/14 - 0s - 8ms/step - accuracy: 0.9861 - loss: 0.0471
Epoch 11/50
14/14 - 0s - 7ms/step - accuracy: 0.9861 - loss: 0.0432
Epoch 12/50
14/14 - 0s - 8ms/step - accuracy: 0.9907 - loss: 0.0395
Epoch 13/50
14/14 - 0s - 7ms/step - accuracy: 0.9907 - loss: 0.0362
Epoch 14/50
14/14 - 0s - 8ms/step - accuracy: 0.9907 - loss: 0.0331
Epoch 15/50
14/14 - 0s - 7ms/step - accuracy: 0.9931 

In [9]:
model.save("../models/breast_cancer_model.keras")

In [10]:
import tensorflow as tf
import joblib
import numpy as np
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

model = tf.keras.models.load_model("../models/breast_cancer_model.keras")
scaler = joblib.load("../models/scaler.joblib")

class PredictRequest(BaseModel):
    features: list[float]  # los 30 valores, en el mismo orden que las columnas del CSV

@app.post("/predict")
def predict(req: PredictRequest):
    X = scaler.transform([req.features])
    prob = float(model.predict(X)[0][0])
    label = "benigno" if prob > 0.5 else "maligno"
    return {"prediction": label, "probability": prob}